# 📚 ANOVA: Comparing Three or More Groups at Once
> Section 9 of the Statistics & A/B Testing curriculum
> 통계 & A/B 테스트 커리큘럼 섹션 9

---
# 🎯 Learning Objective (학습 목표)
Today I want to learn / 오늘 배우고 싶은 것:
- [x] Run a one-way ANOVA to compare 3+ group means, and explain why it's safer than many pairwise t-tests<br>일원배치 ANOVA로 3개 이상 그룹 평균을 비교하고, 여러 쌍별 t-검정보다 안전한 이유 설명하기
- [x] Explain what the F-statistic measures in terms of between-group vs within-group variance<br>F-통계량이 그룹 간 분산 대 그룹 내 분산 측면에서 무엇을 측정하는지 설명하기
- [x] Use Tukey's HSD to find which specific groups differ after a significant ANOVA result<br>유의미한 ANOVA 결과 이후 Tukey's HSD로 구체적으로 어느 그룹이 다른지 찾기

---
# 🧠 Concept (개념)

## What is it? (이것은 무엇인가?)
*(Explain it in your own words.)*

**EN:** ANOVA (Analysis of Variance) extends the t-test's two-group comparison to THREE OR MORE groups at once. Despite the name "analysis of VARIANCE," it's still fundamentally testing whether group MEANS differ — it does this by comparing how much groups vary from each other (between-group variance) to how much individual observations vary within each group (within-group variance).

**KR:** ANOVA(분산분석)는 t-검정의 두 그룹 비교를 세 개 이상의 그룹으로 한 번에 확장합니다. "분산 분석"이라는 이름에도 불구하고, 여전히 근본적으로는 그룹 평균이 다른지를 검정합니다 -- 그룹들이 서로 얼마나 다른지(그룹 간 분산)를, 각 그룹 내 개별 관측치가 얼마나 다른지(그룹 내 분산)와 비교하는 방식으로 이를 수행합니다.

### Why not just run pairwise t-tests? (왜 그냥 쌍별 t-검정을 돌리지 않는가?)

| Number of groups<br>그룹 수 | Pairwise t-tests needed<br>필요한 쌍별 t-검정 수 | Chance of >=1 false positive (each at alpha=0.05)<br>오탐 1개 이상 발생 확률 |
|---|---|---|
| 2 | 1 | ~5% |
| 3 | 3 | ~14.3% |
| 4 | 6 | ~26.5% |
| 5 | 10 | ~40.1% |
| 10 | 45 | ~90.1% |

*The more groups you compare pairwise, the more likely you are to find a "significant" difference purely by chance — even if nothing is really different.*
*쌍별로 비교하는 그룹이 많아질수록, 실제로는 아무 차이가 없어도 순전히 우연으로 "유의미한" 차이를 발견할 가능성이 높아집니다.*

## Why do we use it? (왜 사용하는가?)
*(When is it useful?)*

**EN:** You might think "just run a t-test on every pair of groups" would work for comparing, say, 4 marketing channels — but with 4 groups that's 6 separate t-tests, and each one carries a 5% false-positive risk on its own. Run enough pairwise tests and you're almost guaranteed to find a "significant" difference somewhere by pure chance. ANOVA solves this by asking one combined question ("does ANY group differ?") with one controlled error rate.

**KR:** 예를 들어 마케팅 채널 4개를 비교할 때 "모든 그룹 쌍에 t-검정을 돌리면 되지 않나"라고 생각할 수 있지만, 그룹이 4개면 별도의 t-검정이 6번 필요하고, 각각 독립적으로 5%의 오탐 위험을 갖습니다. 충분히 많은 쌍별 검정을 돌리면 순전히 우연으로 어딘가에서 "유의미한" 차이를 발견할 가능성이 거의 확실해집니다. ANOVA는 하나의 결합된 질문("어떤 그룹이든 다른가?")을 하나의 통제된 오류율로 물어 이를 해결합니다.

## When is it used in Business Analytics? (비즈니스 분석에서 언제 사용되는가?)
*(Real-world use case)*

**EN:** Comparing average conversion or revenue across 3+ marketing channels, regions, or product variants at once, testing whether a multi-arm (A/B/C/D) experiment has any winner before drilling into which specific arm, and any "compare more than two groups" situation that would otherwise tempt you into a pile of separate t-tests.

**KR:** 마케팅 채널·지역·제품 변형 3개 이상의 평균 전환·매출을 한 번에 비교, 여러 그룹(A/B/C/D) 실험에서 어떤 우승자가 있는지 먼저 검정한 뒤 구체적으로 어느 그룹인지 파고들기, 그리고 여러 개의 별도 t-검정을 돌리고 싶은 유혹이 드는 "두 개 이상 그룹 비교" 상황 전반.

---
# 📝 Syntax (문법)

## Basic Syntax (기본 문법)

In [9]:
import numpy as np
import pandas as pd
from scipy import stats

# One-way ANOVA: compare average order value across 4 marketing channels
# 일원배치 ANOVA: 4개 마케팅 채널 간 평균 주문금액 비교
np.random.seed(42)
channel_a = np.random.normal(50, 10, 60)
channel_b = np.random.normal(53, 10, 60)
channel_c = np.random.normal(58, 10, 60)
channel_d = np.random.normal(51, 10, 60)

f_stat, p_value = stats.f_oneway(channel_a, channel_b, channel_c, channel_d)

print(f"F-statistic: {f_stat:.3f}")
print(f"p-value: {p_value:.4f}")

F-statistic: 12.218
p-value: 0.0000


## Common Variations (자주 쓰는 변형)

In [10]:
# Organize as a tidy DataFrame -- most tools in this section work better this way
# tidy DataFrame으로 정리 -- 이 섹션의 도구 대부분이 이 형태에서 더 잘 작동함
df = pd.DataFrame({
    'value': np.concatenate([channel_a, channel_b, channel_c, channel_d]),
    'channel': ['A'] * 60 + ['B'] * 60 + ['C'] * 60 + ['D'] * 60
})
print(df.groupby('channel')['value'].agg(['mean', 'std', 'count']))

              mean        std  count
channel                             
A        48.453453   9.085211     60
B        52.963166   9.433632     60
C        58.919143   9.961032     60
D        51.568143  10.363611     60


---
# 🧪 Small Examples (예제)

## Example 1 — One-Way ANOVA: Comparing 4 Channels, and Why Not Pairwise t-tests (4개 채널 비교, 왜 쌍별 t-검정이 아닌가)

In [11]:
# Business question: does average revenue-per-visitor differ across 4
# marketing channels? Why not just run 6 separate t-tests?
# 비즈니스 질문: 4개 마케팅 채널 간 방문자당 평균 매출이 다른가?
# 왜 그냥 6번의 별도 t-검정을 돌리지 않는가?

import itertools

np.random.seed(1)
channels = {
    'Organic': np.random.normal(12, 4, 80),
    'Paid Search': np.random.normal(14, 4, 80),
    'Social': np.random.normal(11, 4, 80),
    'Email': np.random.normal(15, 4, 80),
}

f_stat, p_value = stats.f_oneway(*channels.values())
print(f"One-way ANOVA: F={f_stat:.3f}, p={p_value:.4f}")
print()

pairs = list(itertools.combinations(channels.keys(), 2))
print(f"Comparing {len(channels)} channels pairwise would need {len(pairs)} separate t-tests:")
for pair in pairs:
    print(f"  {pair[0]} vs {pair[1]}")

n_tests = len(pairs)
false_positive_chance = 1 - (1 - 0.05) ** n_tests
print(f"\nEven if NONE of the channels truly differ, running {n_tests} separate tests at")
print(f"alpha=0.05 gives roughly a {false_positive_chance:.1%} chance of a false 'significant'")
print("result somewhere just by chance. ANOVA avoids this by asking ONE question.")
print(f"채널이 실제로 전혀 다르지 않아도, alpha=0.05로 {n_tests}번의 개별 검정을 돌리면")
print("순전히 우연으로 어딘가에서 잘못된 '유의미한' 결과가 나올 확률이 약")
print(f"{false_positive_chance:.1%}입니다. ANOVA는 질문 하나로 물어 이를 피합니다.")

One-way ANOVA: F=16.632, p=0.0000

Comparing 4 channels pairwise would need 6 separate t-tests:
  Organic vs Paid Search
  Organic vs Social
  Organic vs Email
  Paid Search vs Social
  Paid Search vs Email
  Social vs Email

Even if NONE of the channels truly differ, running 6 separate tests at
alpha=0.05 gives roughly a 26.5% chance of a false 'significant'
result somewhere just by chance. ANOVA avoids this by asking ONE question.
채널이 실제로 전혀 다르지 않아도, alpha=0.05로 6번의 개별 검정을 돌리면
순전히 우연으로 어딘가에서 잘못된 '유의미한' 결과가 나올 확률이 약
26.5%입니다. ANOVA는 질문 하나로 물어 이를 피합니다.


## Example 2 — The F-statistic: Between-Group vs Within-Group Variance (그룹 간 분산 vs 그룹 내 분산)

In [12]:
# Business question: what does the F-statistic actually measure? Compare two
# scenarios with the SAME group-mean gaps but different within-group spread.
# 비즈니스 질문: F-통계량은 실제로 무엇을 측정하는가? 그룹 평균 차이는
# 같지만 그룹 내 퍼짐이 다른 두 시나리오를 비교해보자.

np.random.seed(2)
# Scenario A: same mean gaps, but TIGHT within-group spread (easy to tell groups apart)
tight_a = np.random.normal(50, 2, 40)
tight_b = np.random.normal(55, 2, 40)
tight_c = np.random.normal(60, 2, 40)

# Scenario B: same mean gaps, but WIDE within-group spread (hard to tell groups apart)
wide_a = np.random.normal(50, 15, 40)
wide_b = np.random.normal(55, 15, 40)
wide_c = np.random.normal(60, 15, 40)

f_tight, p_tight = stats.f_oneway(tight_a, tight_b, tight_c)
f_wide, p_wide = stats.f_oneway(wide_a, wide_b, wide_c)

print("Scenario A (tight within-group spread):")
print(f"  Group means: {tight_a.mean():.1f}, {tight_b.mean():.1f}, {tight_c.mean():.1f}")
print(f"  F-statistic: {f_tight:.2f}, p-value: {p_tight:.2e}")
print()
print("Scenario B (wide within-group spread):")
print(f"  Group means: {wide_a.mean():.1f}, {wide_b.mean():.1f}, {wide_c.mean():.1f}")
print(f"  F-statistic: {f_wide:.2f}, p-value: {p_wide:.4f}")
print()
print("Both scenarios have SIMILAR group-mean gaps (~5 points apart), but Scenario A's")
print("F-statistic is far larger. F = between-group variance / within-group variance --")
print("the SAME mean differences are much more convincing when individual observations")
print("are tightly clustered (less 'noise' to explain away the pattern).")
print("두 시나리오 모두 그룹 평균 차이는 비슷하지만(약 5점 차이), 시나리오 A의")
print("F-통계량이 훨씬 큽니다. F = 그룹 간 분산 / 그룹 내 분산 -- 개별 관측치가")
print("촘촘히 모여 있을수록(패턴을 설명해버릴 '노이즈'가 적을수록) 같은 평균")
print("차이라도 훨씬 설득력 있게 보입니다.")

Scenario A (tight within-group spread):
  Group means: 49.6, 54.7, 60.2
  F-statistic: 249.39, p-value: 6.42e-43

Scenario B (wide within-group spread):
  Group means: 51.4, 58.3, 58.5
  F-statistic: 2.50, p-value: 0.0864

Both scenarios have SIMILAR group-mean gaps (~5 points apart), but Scenario A's
F-statistic is far larger. F = between-group variance / within-group variance --
the SAME mean differences are much more convincing when individual observations
are tightly clustered (less 'noise' to explain away the pattern).
두 시나리오 모두 그룹 평균 차이는 비슷하지만(약 5점 차이), 시나리오 A의
F-통계량이 훨씬 큽니다. F = 그룹 간 분산 / 그룹 내 분산 -- 개별 관측치가
촘촘히 모여 있을수록(패턴을 설명해버릴 '노이즈'가 적을수록) 같은 평균
차이라도 훨씬 설득력 있게 보입니다.


## Example 3 — Post-hoc Tests: Tukey's HSD to Find WHICH Groups Differ (어느 그룹이 다른지 찾는 사후 검정)

In [13]:
# Business question: ANOVA found a significant difference SOMEWHERE among 4
# regions' average satisfaction scores. Which SPECIFIC region pairs differ?
# 비즈니스 질문: ANOVA 결과 4개 지역의 평균 만족도 점수 어딘가에 유의미한
# 차이가 발견됐습니다. 구체적으로 어느 지역 쌍이 다른가?

from scipy.stats import tukey_hsd

np.random.seed(3)
seoul = np.random.normal(4.2, 0.5, 50)
busan = np.random.normal(4.0, 0.5, 50)
daegu = np.random.normal(3.7, 0.5, 50)
incheon = np.random.normal(4.3, 0.5, 50)

f_stat, p_value = stats.f_oneway(seoul, busan, daegu, incheon)
print(f"ANOVA: F={f_stat:.3f}, p={p_value:.4f}")
print("-> ANOVA tells us SOME region differs, but not which. Running Tukey's HSD:")
print()

labels = ['Seoul', 'Busan', 'Daegu', 'Incheon']
result = tukey_hsd(seoul, busan, daegu, incheon)
print(result)

print("\nPairs that differ significantly (p < 0.05):")
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        p = result.pvalue[i, j]
        tag = "SIGNIFICANT" if p < 0.05 else "not significant"
        print(f"  {labels[i]} vs {labels[j]}: p={p:.4f} -- {tag}")

ANOVA: F=10.403, p=0.0000
-> ANOVA tells us SOME region differs, but not which. Running Tukey's HSD:

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)      0.015     0.999    -0.238     0.268
 (0 - 2)      0.338     0.004     0.086     0.591
 (0 - 3)     -0.199     0.176    -0.452     0.053
 (1 - 0)     -0.015     0.999    -0.268     0.238
 (1 - 2)      0.324     0.006     0.071     0.577
 (1 - 3)     -0.214     0.128    -0.467     0.039
 (2 - 0)     -0.338     0.004    -0.591    -0.086
 (2 - 1)     -0.324     0.006    -0.577    -0.071
 (2 - 3)     -0.538     0.000    -0.791    -0.285
 (3 - 0)      0.199     0.176    -0.053     0.452
 (3 - 1)      0.214     0.128    -0.039     0.467
 (3 - 2)      0.538     0.000     0.285     0.791


Pairs that differ significantly (p < 0.05):
  Seoul vs Busan: p=0.9988 -- not significant
  Seoul vs Daegu: p=0.0036 -- SIGNIFICANT
  Seoul vs Incheon: p=0.1758 -- not significant
  Busan vs

## Example 4 — ANOVA Assumptions and the Bridge to Kruskal-Wallis (ANOVA의 가정과 Kruskal-Wallis로의 연결)

In [14]:
# Business question: before trusting an ANOVA comparing response times across
# 3 support channels, are its assumptions actually met?
# 비즈니스 질문: 3개 지원 채널의 응답 시간을 비교하는 ANOVA를 신뢰하기
# 전에, 그 가정이 실제로 충족되는가?

np.random.seed(4)
chat = np.random.exponential(8, 50)     # skewed, common for wait times
email = np.random.exponential(12, 50)   # skewed
phone = np.random.exponential(6, 50)    # skewed

# 1) Normality check (skewness, from Section 3)
for name, data in [('Chat', chat), ('Email', email), ('Phone', phone)]:
    print(f"{name} skewness: {stats.skew(data):.2f}")
print("All heavily right-skewed -- individual response times are NOT normal.")
print()

# 2) Equal variance check (Levene's test, from Section 7)
levene_stat, levene_p = stats.levene(chat, email, phone)
print(f"Levene's test: p={levene_p:.4f}")
if levene_p < 0.05:
    print("Variances are significantly different -- standard ANOVA's equal-variance")
    print("assumption is violated.")

# 3) Compare standard ANOVA to the non-parametric alternative: Kruskal-Wallis
f_stat, anova_p = stats.f_oneway(chat, email, phone)
h_stat, kw_p = stats.kruskal(chat, email, phone)

print(f"\nStandard ANOVA:  F={f_stat:.3f}, p={anova_p:.4f}")
print(f"Kruskal-Wallis:  H={h_stat:.3f}, p={kw_p:.4f}")
print()
print("When assumptions are shaky, Kruskal-Wallis (a rank-based test, like Spearman was")
print("to Pearson in Section 5) is the safer choice -- covered in full next section.")
print("가정이 불안정할 때는 Kruskal-Wallis(섹션 5에서 Spearman이 Pearson에 대해")
print("그랬던 것처럼 순위 기반 검정)가 더 안전한 선택입니다 -- 다음 섹션에서")
print("자세히 다룹니다.")

Chat skewness: 1.41
Email skewness: 0.85
Phone skewness: 1.34
All heavily right-skewed -- individual response times are NOT normal.

Levene's test: p=0.4432

Standard ANOVA:  F=3.331, p=0.0385
Kruskal-Wallis:  H=8.414, p=0.0149

When assumptions are shaky, Kruskal-Wallis (a rank-based test, like Spearman was
to Pearson in Section 5) is the safer choice -- covered in full next section.
가정이 불안정할 때는 Kruskal-Wallis(섹션 5에서 Spearman이 Pearson에 대해
그랬던 것처럼 순위 기반 검정)가 더 안전한 선택입니다 -- 다음 섹션에서
자세히 다룹니다.


## Example 5 — Practice: Apply All Four Concepts (연습: 네 가지 개념 모두 적용하기)

Fill in every `________` blank below. The cell runs with **no errors** even if you leave blanks — it just prints a reminder instead of that part's result.

아래 `________` 빈칸을 모두 채워보세요. 빈칸을 두어도 셀은 **에러 없이** 실행되며, 그 부분의 결과 대신 안내 메시지만 출력됩니다.

In [15]:
# ============================================================
# Practice -- fill in every ________ blank, then re-run the cell
# 연습 -- 아래 ________ 빈칸을 모두 채운 뒤 셀을 다시 실행하세요
# ============================================================
BLANK = "________"

def ready(*vals):
    return all(v != BLANK for v in vals)

# ---------- Part A: one-way ANOVA -- three store layouts' average basket size ----------
np.random.seed(5)
layout_1 = np.random.normal(42, 8, 50)
layout_2 = np.random.normal(45, 8, 50)
layout_3 = np.random.normal(48, 8, 50)

test_func_a = 'f_oneway'   # TODO: which scipy.stats function compares 3+ group means at once? ('f_oneway')

if ready(test_func_a):
    f_a, p_a = getattr(stats, test_func_a)(layout_1, layout_2, layout_3)
    print(f"F={f_a:.3f}, p={p_a:.4f}")
else:
    print("Part A \u270f\ufe0f  Fill in test_func_a above, then re-run.")

# ---------- Part B: F-statistic intuition -- predict which scenario has a bigger F ----------
np.random.seed(6)
narrow_x = np.random.normal(100, 3, 30)
narrow_y = np.random.normal(103, 3, 30)
wide_x = np.random.normal(100, 20, 30)
wide_y = np.random.normal(103, 20, 30)

bigger_f_scenario_b = 'narrow'   # TODO: 'narrow' or 'wide' -- which pair should produce the
                              # LARGER F-statistic, given both have the SAME 3-point mean gap?

if ready(bigger_f_scenario_b):
    f_narrow, _ = stats.f_oneway(narrow_x, narrow_y)
    f_wide, _ = stats.f_oneway(wide_x, wide_y)
    print(f"narrow scenario F={f_narrow:.2f}, wide scenario F={f_wide:.2f}")
    correct = (bigger_f_scenario_b == 'narrow') == (f_narrow > f_wide)
    if correct:
        print("Correct!")
    else:
        print("Not quite -- tighter spread means a bigger F for the same mean gap.")
else:
    print("Part B \u270f\ufe0f  Fill in bigger_f_scenario_b above ('narrow' or 'wide'), then re-run.")

# ---------- Part C: post-hoc -- Tukey's HSD on four ad creatives ----------
np.random.seed(7)
creative_1 = np.random.normal(2.0, 0.5, 60)
creative_2 = np.random.normal(2.6, 0.5, 60)
creative_3 = np.random.normal(2.1, 0.5, 60)
creative_4 = np.random.normal(1.9, 0.5, 60)

posthoc_func_c = 'tukey_hsd'   # TODO: which scipy.stats function finds WHICH specific groups
                         # differ after a significant ANOVA? ('tukey_hsd')

if ready(posthoc_func_c):
    result_c = getattr(stats, posthoc_func_c)(creative_1, creative_2, creative_3, creative_4)
    print(result_c)
else:
    print("Part C \u270f\ufe0f  Fill in posthoc_func_c above, then re-run.")

# ---------- Part D: assumption check -- decide ANOVA vs Kruskal-Wallis ----------
np.random.seed(8)
tiny_group_1 = np.random.exponential(5, 8)   # small, skewed
tiny_group_2 = np.random.exponential(9, 8)   # small, skewed
tiny_group_3 = np.random.exponential(4, 8)   # small, skewed

use_kruskal_d = True   # TODO: True or False -- given these are SMALL (n=8) and SKEWED
                        # (exponential) groups, should you prefer Kruskal-Wallis over ANOVA?

if ready(use_kruskal_d):
    print(f"Your choice: use Kruskal-Wallis = {use_kruskal_d}")
    if use_kruskal_d:
        h_d, p_d = stats.kruskal(tiny_group_1, tiny_group_2, tiny_group_3)
        print(f"Kruskal-Wallis: H={h_d:.3f}, p={p_d:.4f}")
        print("Correct -- small + skewed groups are exactly when the rank-based test is safer.")
    else:
        print("Reconsider -- small, skewed groups are exactly when ANOVA's assumptions are shakiest.")
else:
    print("Part D \u270f\ufe0f  Fill in use_kruskal_d above (True or False), then re-run.")

F=4.958, p=0.0082
narrow scenario F=8.20, wide scenario F=0.40
Correct!
Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)     -0.672     0.000    -0.898    -0.445
 (0 - 2)     -0.047     0.951    -0.273     0.180
 (0 - 3)      0.086     0.759    -0.140     0.313
 (1 - 0)      0.672     0.000     0.445     0.898
 (1 - 2)      0.625     0.000     0.398     0.852
 (1 - 3)      0.758     0.000     0.531     0.984
 (2 - 0)      0.047     0.951    -0.180     0.273
 (2 - 1)     -0.625     0.000    -0.852    -0.398
 (2 - 3)      0.133     0.430    -0.094     0.359
 (3 - 0)     -0.086     0.759    -0.313     0.140
 (3 - 1)     -0.758     0.000    -0.984    -0.531
 (3 - 2)     -0.133     0.430    -0.359     0.094

Your choice: use Kruskal-Wallis = True
Kruskal-Wallis: H=3.545, p=0.1699
Correct -- small + skewed groups are exactly when the rank-based test is safer.


---
# ⚠️ Common Mistakes (자주 하는 실수)

### Mistake 1 — Running Many Pairwise t-tests Instead of ANOVA (ANOVA 대신 여러 개의 쌍별 t-검정 실행)
Comparing k groups with C(k,2) separate t-tests inflates your overall Type I error rate far beyond your chosen alpha — with just 4 groups (6 tests), the chance of at least one false "significant" result climbs to roughly 26%, not 5%.  
k개 그룹을 C(k,2)개의 별도 t-검정으로 비교하면 전체 1종 오류율이 선택한 alpha를 훨씬 넘어 부풀려집니다 -- 그룹이 4개(검정 6번)만 되어도 최소 하나의 오탐 "유의미한" 결과가 나올 확률이 5%가 아니라 약 26%까지 올라갑니다.

✅ **Fix / 해결:** Use one-way ANOVA to test all groups at once with a single, controlled error rate.

### Mistake 2 — Assuming a Significant ANOVA Tells You WHICH Groups Differ (유의미한 ANOVA가 어느 그룹이 다른지 알려준다고 착각)
ANOVA's p-value only answers "does at least one group differ from the others?" — it says nothing about which specific pair (or pairs) are responsible, exactly like a significant chi-square test of independence (Section 8) doesn't identify which cells drive it.  
ANOVA의 p-value는 "적어도 하나의 그룹이 다른가?"에만 답합니다 -- 어느 특정 쌍(들)이 원인인지는 알려주지 않습니다. 유의미한 카이제곱 독립성 검정(섹션 8)이 어느 셀이 원인인지 알려주지 않는 것과 정확히 같습니다.

✅ **Fix / 해결:** Follow a significant ANOVA with a post-hoc test (Tukey's HSD) to identify the specific differing pairs.

### Mistake 3 — Running Plain t-tests as the "Post-hoc" Step (사후 분석 단계로 일반 t-검정 사용)
After a significant ANOVA, running ordinary pairwise t-tests to find "which groups differ" reintroduces the exact multiple-comparisons problem ANOVA was meant to avoid in the first place.
유의미한 ANOVA 이후 "어느 그룹이 다른지" 찾기 위해 일반적인 쌍별 t-검정을 돌리면, 애초에 ANOVA로 피하려던 다중비교 문제를 다시 끌어들이게 됩니다.

✅ **Fix / 해결:** Use a correction designed for this exact situation — Tukey's HSD adjusts the threshold specifically to control the error rate across all pairwise comparisons.

### Mistake 4 — Confusing a Large F-statistic with a Large, Important Effect (큰 F-통계량을 크고 중요한 효과로 착각)
Just like Section 6's statistical-vs-practical-significance warning, a large F-statistic (driven by a big sample or tiny within-group variance) can accompany a business-meaningless difference between group means.  
섹션 6의 통계적 대 실질적 유의성 경고와 마찬가지로, 큰 F-통계량(큰 표본이나 아주 작은 그룹 내 분산 때문에)이 비즈니스적으로 무의미한 그룹 평균 차이와 함께 나타날 수 있습니다.

✅ **Fix / 해결:** Report the actual group means (and their differences) alongside the F-statistic, not the F-statistic alone.

### Mistake 5 — Ignoring the Equal-Variance Assumption Across 3+ Groups (3개 이상 그룹의 등분산 가정 무시)
Standard one-way ANOVA assumes all groups share roughly the same variance — with 3+ groups, it's even easier to have one unusually volatile group than with just two (Section 7's t-test concern), which can distort the result.  
표준 일원배치 ANOVA는 모든 그룹이 대략 같은 분산을 공유한다고 가정합니다 -- 그룹이 3개 이상이면, 그룹이 둘일 때(섹션 7의 t-검정 관심사)보다 유별나게 변동성이 큰 그룹 하나가 있을 가능성이 더 큽니다.

✅ **Fix / 해결:** Run Levene's test across all groups (Example 4) before trusting standard ANOVA; consider Welch's ANOVA variant if variances clearly differ.

### Mistake 6 — Defaulting to ANOVA on Small, Heavily Skewed Samples (작고 심하게 치우친 표본에 기본적으로 ANOVA 사용)
Just like the t-test, ANOVA's p-value relies on the Central Limit Theorem to handle non-normal raw data — with small groups and severe skew, that safety net weakens, and the result can be unreliable.  
t-검정과 마찬가지로, ANOVA의 p-value는 비정규 원본 데이터를 다루기 위해 중심극한정리에 의존합니다 -- 그룹이 작고 왜도가 심하면 그 안전망이 약해져 결과를 신뢰할 수 없게 됩니다.

✅ **Fix / 해결:** Check skewness and sample size per group first; switch to Kruskal-Wallis (next section) when in doubt.

---
# 💡 Tips (팁)
Useful tips or shortcuts / 유용한 팁과 단축법

- `stats.f_oneway()` accepts any number of groups as separate arguments — you're not limited to 3; it scales to as many groups as your business question needs.<br>stats.f_oneway()는 몇 개의 그룹이든 별도의 인자로 받을 수 있습니다 -- 3개로 제한되지 않으며, 비즈니스 질문에 필요한 만큼의 그룹으로 확장됩니다.
- The multiple-comparisons math (`1 - (1-alpha)^k`) is worth memorizing as a rough mental check — even 3 tests already pushes your real error rate to about 14%, nearly triple your intended 5%.<br>다중비교 계산식(1 - (1-alpha)^k)은 빠른 암산 점검으로 기억해둘 가치가 있습니다 -- 검정이 단 3번만 되어도 실제 오류율은 이미 약 14%로, 의도한 5%의 거의 3배가 됩니다.
- Tukey's HSD is specifically built for "compare every pair after ANOVA" — it's not just a t-test with a stricter alpha; the math is designed around this exact use case.<br>Tukey's HSD는 특히 "ANOVA 이후 모든 쌍 비교"를 위해 만들어졌습니다 -- 단순히 alpha를 더 엄격하게 한 t-검정이 아니라, 이 정확한 사용 사례에 맞춰 설계된 수학입니다.
- ANOVA with exactly 2 groups gives mathematically the same p-value as an independent t-test on those same 2 groups (F = t²) — a useful sanity check if you ever need to confirm the formulas connect.<br>정확히 2개 그룹에 ANOVA를 적용하면 같은 두 그룹의 독립표본 t-검정과 수학적으로 같은 p-value를 줍니다(F = t²) -- 공식들이 어떻게 연결되는지 확인하고 싶을 때 유용한 점검법입니다.
- Group your data into a "tidy" long-format DataFrame (one row per observation, a column for the group label) early — nearly every tool in this section works better with this shape than with separate arrays.<br>데이터를 일찍부터 "tidy" 롱 포맷 DataFrame으로 정리하세요(관측치당 한 행, 그룹 라벨용 컬럼 하나) -- 이 섹션의 거의 모든 도구가 별도 배열보다 이 형태에서 더 잘 작동합니다.

---
# 🔗 Related Concepts (관련 개념)

```
Chi-Square & Categorical Tests (Section 8)
   goodness of fit, test of independence, Fisher's exact test
        ↓
ANOVA (Section 9)  ← you are here / 현재 위치
   one-way ANOVA, F-statistic, Tukey's HSD, assumptions
        ↓
Non-Parametric Tests (Section 10)
   Kruskal-Wallis (ANOVA's rank-based alternative), Mann-Whitney, Wilcoxon
        ↓
[ Part 3: Regression ] -> [ Part 4: A/B Testing in Practice ]
```

*How is today's topic connected to other concepts?*

**EN:** ANOVA is what a t-test becomes once you have 3 or more groups instead of 2 — in fact, running ANOVA on exactly 2 groups gives the identical p-value as Section 7's independent t-test. The multiple-comparisons problem from Example 1 is not a one-time concern: it resurfaces directly in later A/B testing sections whenever a test has more than two variants (A/B/C/D tests) or checks more than one metric at once, and the same family of corrections (Tukey's HSD here, similar corrections there) is the standard fix. And when this section's assumptions don't hold, Section 10's Kruskal-Wallis test is waiting as the direct rank-based substitute — the same relationship Spearman has to Pearson.

**KR:** ANOVA는 그룹이 2개가 아니라 3개 이상이 될 때의 t-검정입니다 -- 실제로 정확히 2개 그룹에 ANOVA를 돌리면 섹션 7의 독립표본 t-검정과 동일한 p-value가 나옵니다. 예제 1의 다중비교 문제는 일회성 걱정이 아닙니다: 변형이 두 개보다 많은 실험(A/B/C/D 테스트)이나 여러 지표를 한 번에 확인하는 경우 이 커리큘럼 후반의 A/B 테스트 섹션에서 직접 다시 등장하며, 같은 계열의 보정(여기서는 Tukey's HSD, 거기서는 유사한 보정)이 표준 해결책입니다. 그리고 이 섹션의 가정이 성립하지 않을 때는 섹션 10의 Kruskal-Wallis 검정이 직접적인 순위 기반 대체재로 기다리고 있습니다 -- Spearman이 Pearson에 대해 갖는 것과 같은 관계입니다.

---
# 💼 Business Example (비즈니스 예시)
*How would a Business Analyst use this?*

**Scenario / 시나리오:**

**EN:** Your team ran a 4-way test (A/B/C/D) on landing page designs, measuring average time-to-signup. Leadership wants to know: is there a winner among the 4, which specific design(s) actually differ from the baseline (Variant A), and can the result be trusted given the sample sizes involved?

**KR:** 팀이 랜딩 페이지 디자인 4가지(A/B/C/D)에 대해 평균 가입까지 걸리는 시간을 측정하는 4-way 테스트를 진행했습니다. 경영진은 알고 싶어합니다: 4개 중 우승자가 있는가, 어느 디자인이 기준(변형 A)과 실제로 다른가, 그리고 관련된 표본 크기를 고려했을 때 결과를 신뢰할 수 있는가?

**To-do / 할 일:**
- [x] Check the equal-variance assumption before trusting standard ANOVA<br>표준 ANOVA를 신뢰하기 전에 등분산 가정 확인하기
- [x] Run one-way ANOVA across all 4 variants<br>4개 변형 전체에 일원배치 ANOVA 돌리기
- [x] If significant, run Tukey's HSD to compare each variant against the baseline<br>유의미하면 Tukey's HSD로 각 변형을 기준과 비교하기
- [x] Write a recommendation for which variant to prioritize<br>어느 변형을 우선할지 제안 작성하기

In [16]:
from scipy.stats import tukey_hsd

np.random.seed(40)
variant_a = np.random.normal(95, 20, 70)  # baseline
variant_b = np.random.normal(88, 20, 70)
variant_c = np.random.normal(78, 20, 70)  # meaningfully faster
variant_d = np.random.normal(92, 20, 70)

# Step 1: check assumptions
levene_stat, levene_p = stats.levene(variant_a, variant_b, variant_c, variant_d)
print(f"Levene's test for equal variance: p={levene_p:.4f}")
print("Assumption OK" if levene_p >= 0.05 else "Assumption violated -- interpret with caution")

# Step 2: one-way ANOVA
f_stat, anova_p = stats.f_oneway(variant_a, variant_b, variant_c, variant_d)
print(f"\nANOVA: F={f_stat:.3f}, p={anova_p:.6f}")

# Step 3: post-hoc if significant
labels = ['A (baseline)', 'B', 'C', 'D']
means = [variant_a.mean(), variant_b.mean(), variant_c.mean(), variant_d.mean()]
if anova_p < 0.05:
    print("-> significant overall difference found; running Tukey's HSD to locate it")
    result = tukey_hsd(variant_a, variant_b, variant_c, variant_d)
    print()
    for i, label in enumerate(labels):
        print(f"{label}: mean={means[i]:.1f}s")
    print()
    print("Comparisons vs baseline (Variant A):")
    for j in range(1, 4):
        p = result.pvalue[0, j]
        sig = "SIGNIFICANT" if p < 0.05 else "not significant"
        print(f"  A vs {labels[j]}: p={p:.4f} -- {sig}")

print()
print("Recommendation: Variant C shows the largest, statistically confirmed improvement")
print("over baseline -- prioritize it for further testing/rollout.")
print()
print("추천: 변형 C가 기준 대비 가장 크고 통계적으로 확인된 개선을 보여줍니다 --")
print("추가 테스트·출시를 위해 우선순위를 두세요.")

Levene's test for equal variance: p=0.4464
Assumption OK

ANOVA: F=7.870, p=0.000047
-> significant overall difference found; running Tukey's HSD to locate it

A (baseline): mean=96.2s
B: mean=84.6s
C: mean=79.9s
D: mean=87.4s

Comparisons vs baseline (Variant A):
  A vs B: p=0.0051 -- SIGNIFICANT
  A vs C: p=0.0000 -- SIGNIFICANT
  A vs D: p=0.0546 -- not significant

Recommendation: Variant C shows the largest, statistically confirmed improvement
over baseline -- prioritize it for further testing/rollout.

추천: 변형 C가 기준 대비 가장 크고 통계적으로 확인된 개선을 보여줍니다 --
추가 테스트·출시를 위해 우선순위를 두세요.


---
# 📝 Summary (요약)
*Write today's concept in 3~5 sentences.*

**EN:** ANOVA extends the t-test to three or more groups at once, testing whether at least one group's mean differs from the others by comparing between-group variance to within-group variance (the F-statistic) — doing this with one combined test avoids the multiple-comparisons trap of running many separate pairwise t-tests, which can inflate the true error rate to 3-5x the intended alpha with just a handful of groups. A significant ANOVA result only proves SOME group differs, not which — Tukey's HSD is the standard post-hoc follow-up, purpose-built to identify specific differing pairs while still controlling the overall error rate. Like the t-test before it, ANOVA relies on roughly equal variance across groups (checked with Levene's test) and reasonably normal-behaving sample means (via the CLT) — when either assumption looks shaky, especially with small or heavily skewed groups, the non-parametric Kruskal-Wallis test covered next is the safer alternative.

**KR:** ANOVA는 t-검정을 세 개 이상의 그룹으로 한 번에 확장해, 그룹 간 분산과 그룹 내 분산을 비교(F-통계량)함으로써 적어도 하나의 그룹 평균이 다른지 검정합니다 -- 이렇게 하나의 결합된 검정으로 하면, 그룹이 몇 개만 있어도 실제 오류율을 의도한 alpha의 3~5배까지 부풀릴 수 있는 여러 번의 별도 쌍별 t-검정이라는 다중비교 함정을 피할 수 있습니다. 유의미한 ANOVA 결과는 어떤 그룹이 다르다는 것만 증명하지, 어느 것인지는 증명하지 않습니다 -- Tukey's HSD가 표준 사후 분석으로, 전체 오류율을 계속 통제하면서 구체적으로 다른 쌍을 찾도록 특별히 설계되었습니다. 이전의 t-검정과 마찬가지로 ANOVA는 그룹 간 대략적인 등분산(Levene 검정으로 확인)과 대체로 정규적으로 행동하는 표본평균(CLT를 통해)에 의존합니다 -- 어느 쪽이든 불안정해 보이면, 특히 그룹이 작거나 심하게 치우쳐 있으면, 다음에 다룰 비모수적 Kruskal-Wallis 검정이 더 안전한 대안입니다.

---
# 📌 One Sentence Summary (한 문장 요약)
Today's topic in ONE sentence. / 오늘의 주제를 한 문장으로.

> **EN:** ANOVA tests whether 3+ group means differ by comparing between-group to within-group variance in one combined test (avoiding the inflated error rate of many separate t-tests), and a significant result needs a post-hoc test like Tukey's HSD to reveal which specific groups actually differ.

> **KR:** ANOVA는 그룹 간 분산과 그룹 내 분산을 하나의 결합된 검정으로 비교해 3개 이상의 그룹 평균이 다른지 검정하며(여러 번의 별도 t-검정이 만드는 부풀려진 오류율을 피함), 유의미한 결과는 구체적으로 어느 그룹이 다른지 밝히기 위해 Tukey's HSD 같은 사후 검정이 필요합니다.

---
# ❓ Review Questions (복습 질문)

**Q1 (EN).** Why does running 6 separate t-tests to compare 4 groups pairwise inflate your true error rate well beyond 5%, even if each individual test uses alpha=0.05?  
**Q1 (KR).** 4개 그룹을 쌍별로 비교하기 위해 6번의 별도 t-검정을 돌리면, 각 개별 검정이 alpha=0.05를 사용해도 왜 실제 오류율이 5%를 훨씬 넘게 부풀려지나요?

Each test has a 5% chance of a false positive, and these chances stack across 6 tests, so the overall error rate is about 26% (1 − 0.95⁶).  
각 검정마다 거짓 양성 확률 5%가 있고, 6번을 돌리면 이 확률이 쌓여 전체 오류율은 약 26%(1 − 0.95⁶)가 됩니다.

**Q2 (EN).** What does the F-statistic actually compare, and why can two scenarios with the same group-mean differences produce very different F-statistics?  
**Q2 (KR).** F-통계량은 실제로 무엇을 비교하며, 그룹 평균 차이가 같은 두 시나리오가 왜 매우 다른 F-통계량을 낼 수 있나요?

F compares the variance between group means with the variance within groups, so the same mean differences give a smaller F when the groups are more spread out.  
F는 그룹 평균 간 분산을 그룹 내 분산과 비교하므로, 평균 차이가 같아도 그룹 내 퍼짐이 크면 F가 작아집니다.

**Q3 (EN).** An ANOVA comes back with p=0.002. Can you conclude Group A specifically differs from Group C? What do you need to run to find out?  
**Q3 (KR).** ANOVA 결과 p=0.002가 나왔습니다. 특히 그룹 A가 그룹 C와 다르다고 결론 내릴 수 있나요? 알아내려면 무엇을 돌려야 하나요?

No. A significant ANOVA only says at least one group differs; run a post-hoc test such as Tukey's HSD to compare A with C.
아니요. 유의한 ANOVA는 적어도 한 그룹이 다르다는 뜻일 뿐이며, A와 C를 비교하려면 Tukey's HSD 같은 사후 검정을 돌립니다.

**Q4 (EN).** Why is running plain t-tests as your "post-hoc" step after a significant ANOVA still problematic, and what does Tukey's HSD do differently?  
**Q4 (KR).** 유의미한 ANOVA 이후 "사후 분석" 단계로 일반 t-검정을 돌리는 것이 여전히 문제가 되는 이유는 무엇이며, Tukey's HSD는 무엇을 다르게 하나요?

Plain t-tests on every pair bring back the stacked-error problem. Tukey's HSD compares all pairs while keeping the overall error rate at 5%.
모든 쌍에 일반 t-검정을 돌리면 오류가 쌓이는 문제가 다시 생깁니다. Tukey's HSD는 모든 쌍을 비교하면서 전체 오류율을 5%로 유지합니다.

**Q5 (EN).** If a Levene's test shows unequal variance across your groups and your samples are small and skewed, what should you consider using instead of standard ANOVA?  
**Q5 (KR).** Levene 검정 결과 그룹 간 분산이 다르고 표본이 작고 치우쳐 있다면, 표준 ANOVA 대신 무엇을 사용할지 고려해야 하나요?

Consider Welch's ANOVA for unequal variances, or the Kruskal-Wallis test when the samples are small and skewed.  
분산이 다르면 Welch ANOVA를, 표본이 작고 치우쳐 있으면 Kruskal-Wallis 검정을 고려합니다.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*